<a href="https://colab.research.google.com/github/Absaibro/ProjetoII/blob/main/Aplicacao_Modelo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aplicação prática do modelo

Após o treinamento e validação do modelo Random Forest, nesta etapa é simulada sua aplicação em um cenário real.

O modelo é aplicado apenas aos alunos com situação **Matriculado**, identificando aqueles com maior probabilidade de evasão.

Os dados apresentados são anonimizados, preservando a identidade dos estudantes conforme a Lei Geral de Proteção de Dados (LGPD).

In [17]:
from google.colab import files

uploaded = files.upload()

Saving modelo_evasao_ads.pkl to modelo_evasao_ads (2).pkl
Saving base_evasao.xlsx to base_evasao (2).xlsx


In [18]:
import pandas as pd
import joblib

# Carrega o pacote do modelo

pacote = joblib.load("modelo_evasao_ads (2).pkl")

modelo_rf = pacote["modelo"]

encoder_turma = pacote["encoder_turma"]

encoder_turno = pacote["encoder_turno"]

encoder_modalidade = pacote["encoder_modalidade"]

variaveis = pacote["variaveis"]

# Carrega a base

df = pd.read_excel("base_evasao.xlsx")

print("Modelo carregado com sucesso!")

Modelo carregado com sucesso!


In [21]:
# Filtra apenas alunos matriculados

ativos = df[
    df["HISTORICO_SITUACAO_ALUNO"] == "Matriculado."
].copy()

print(f"Total de alunos ativos: {len(ativos)}")


Total de alunos ativos: 94


In [22]:
# Seleciona apenas as variáveis utilizadas no treinamento

X_ativos = ativos[variaveis].copy()

print(X_ativos.head())

          TURMA      TURNO  MODALIDADE  TotalAulas  TotalFaltas  \
6    ADS1N26/2   NOITE - N  PRESENCIAL           0            0   
56   ADS6N26/2   NOITE - N  PRESENCIAL         660           85   
57   ADS6N26/2   NOITE - N  PRESENCIAL         600           39   
74   ADS1N26/2   NOITE - N  PRESENCIAL           0            0   
101  ADS6N26/2   NOITE - N  PRESENCIAL         720           27   

     TotalFaltasJustificadas  PercentualFaltas  SaldoDevedor  \
6                          0               NaN       3554.99   
56                         0             12.88       2725.68   
57                         0              6.50       5905.08   
74                         0               NaN          0.00   
101                        0              3.75          0.00   

     ParcelasPendentes  PossuiDivida  
6                    7             1  
56                   0             1  
57                   6             1  
74                   0             0  
101              

In [23]:
X_ativos["TURMA"] = X_ativos["TURMA"].astype(str).str.strip()

X_ativos["TURNO"] = X_ativos["TURNO"].astype(str).str.strip()

X_ativos["MODALIDADE"] = X_ativos["MODALIDADE"].astype(str).str.strip()

In [26]:
# Aplica os encoders

X_ativos["TURMA"] = encoder_turma.transform(X_ativos["TURMA"])

X_ativos["TURNO"] = encoder_turno.transform(X_ativos["TURNO"])

X_ativos["MODALIDADE"] = encoder_modalidade.transform(X_ativos["MODALIDADE"])

In [27]:
# Trata os valores ausentes

X_ativos["PercentualFaltas"] = (
    X_ativos["PercentualFaltas"]
    .fillna(X_ativos["PercentualFaltas"].median())
)

In [28]:
# Aplica o modelo

ativos["Probabilidade_Evasao"] = modelo_rf.predict_proba(
    X_ativos
)[:, 1]

print("Predição realizada com sucesso!")

Predição realizada com sucesso!


In [34]:
def classificar_risco(prob):

    if prob >= 0.90:
        return "Muito Alto"

    elif prob >= 0.70:
        return "Alto"

    elif prob >= 0.50:
        return "Médio"

    else:
        return "Baixo"


ativos["Nivel_Risco"] = ativos["Probabilidade_Evasao"].apply(
    classificar_risco
)

In [35]:
ativos["Aluno"] = [
    f"Aluno {i+1}"
    for i in range(len(ativos))
]

In [36]:
resultado = ativos[[
    "Aluno",
    "Nivel_Risco",
    "Probabilidade_Evasao",
    "PercentualFaltas",
    "SaldoDevedor",
    "ParcelasPendentes",
    "PossuiDivida"
]].copy()

resultado["Probabilidade_Evasao"] = (
    resultado["Probabilidade_Evasao"] * 100
).round(2)

resultado["PercentualFaltas"] = (
    resultado["PercentualFaltas"]
    .fillna(0)
    .round(2)
)

resultado["SaldoDevedor"] = (
    resultado["SaldoDevedor"]
    .round(2)
)

resultado["PossuiDivida"] = resultado["PossuiDivida"].replace({
    1: "Sim",
    0: "Não"
})

resultado = resultado.sort_values(
    by="Probabilidade_Evasao",
    ascending=False
)

display(resultado.head(15))

print("=" * 60)
print("Resumo da aplicação do modelo")
print("=" * 60)

print(f"Total de alunos avaliados: {len(resultado)}")

print("\nDistribuição por nível de risco:")

print(resultado["Nivel_Risco"].value_counts())

,Aluno,Nivel_Risco,Probabilidade_Evasao,PercentualFaltas,SaldoDevedor,ParcelasPendentes,PossuiDivida
129,Aluno 7,Muito Alto,97.00,13.78,0.00,0,Não
631,Aluno 28,Muito Alto,95.00,0.00,6135.73,0,Sim
201,Aluno 18,Muito Alto,94.50,0.00,0.00,0,Não
1040,Aluno 52,Muito Alto,94.50,7.69,0.00,0,Não
1090,Aluno 68,Alto,73.14,0.00,0.00,0,Não
142,Aluno 9,Alto,72.00,26.64,0.00,0,Não
1159,Aluno 89,Alto,71.00,5.79,0.00,0,Não
1169,Aluno 90,Médio,66.50,34.66,0.00,0,Não
992,Aluno 45,Médio,66.00,0.71,0.00,0,Não
1107,Aluno 80,Médio,53.00,0.00,0.00,0,Não


Resumo da aplicação do modelo
Total de alunos avaliados: 94

Distribuição por nível de risco:
Nivel_Risco
Baixo         84
Muito Alto     4
Alto           3
Médio          3
Name: count, dtype: int64


In [37]:
resultado.to_excel(
    "Possiveis_Alunos_Risco_Evasao.xlsx",
    index=False
)

print("Arquivo exportado com sucesso!")

Arquivo exportado com sucesso!


In [38]:
from google.colab import files

files.download("Possiveis_Alunos_Risco_Evasao.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Conclusão da aplicação prática

O modelo treinado foi aplicado aos alunos com situação Matriculado, simulando um cenário real de utilização institucional. Os resultados permitiram identificar estudantes com diferentes níveis de risco de evasão, utilizando variáveis acadêmicas e financeiras previamente definidas. Para preservar a privacidade dos alunos, todas as informações de identificação foram anonimizadas. Essa aplicação demonstra como modelos de Machine Learning podem apoiar instituições de ensino na identificação precoce de estudantes com maior probabilidade de evasão, permitindo ações preventivas e direcionadas.